In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [2]:
import os
import torch
import numpy as np
import pandas as pd
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

In [3]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch version: 2.12.0.dev20260312+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU


In [4]:
# Paths
ASVSPOOF_ROOT = r"C:\deepfake-project\data\asvspoof"
PROTOCOL_DIR  = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_cm_protocols")
TRAIN_AUDIO   = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_train", "flac")
DEV_AUDIO     = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_dev", "flac")

# Read train and dev protocol files
train_df = pd.read_csv(
    os.path.join(PROTOCOL_DIR, "ASVspoof2019.LA.cm.train.trn.txt"),
    sep=" ", header=None,
    names=["speaker", "file_id", "env", "attack", "label"]
)

dev_df = pd.read_csv(
    os.path.join(PROTOCOL_DIR, "ASVspoof2019.LA.cm.dev.trl.txt"),
    sep=" ", header=None,
    names=["speaker", "file_id", "env", "attack", "label"]
)

# Convert labels to integers — 0 = bonafide, 1 = spoof
train_df["label"] = train_df["label"].map({"bonafide": 0, "spoof": 1})
dev_df["label"]   = dev_df["label"].map({"bonafide": 0, "spoof": 1})

# Add the full file path for each audio file
train_df["path"] = train_df["file_id"].apply(lambda x: os.path.join(TRAIN_AUDIO, f"{x}.flac"))
dev_df["path"]   = dev_df["file_id"].apply(lambda x: os.path.join(DEV_AUDIO,   f"{x}.flac"))

print(f"Training samples:   {len(train_df)}")
print(f"Dev samples:        {len(dev_df)}")
print(f"Train label split:  {train_df['label'].value_counts().to_dict()}")
print(f"Dev label split:    {dev_df['label'].value_counts().to_dict()}")

Training samples:   25380
Dev samples:        24844
Train label split:  {1: 22800, 0: 2580}
Dev label split:    {1: 22296, 0: 2548}


In [5]:
# Audio standard
SAMPLE_RATE = 16000
MAX_SAMPLES = 64000  # 4 seconds

class ASVspoofDataset(Dataset):
    
    def __init__(self, df, feature_extractor):
        self.df = df
        self.feature_extractor = feature_extractor
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load the audio file
        audio, sr = sf.read(row["path"])
        
        # Convert to float32 if needed
        audio = audio.astype(np.float32)
        
        # Pad or truncate to exactly 4 seconds
        if len(audio) >= MAX_SAMPLES:
            audio = audio[:MAX_SAMPLES]
        else:
            audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))
        
        # Normalise amplitude
        peak = np.abs(audio).max()
        if peak > 0:
            audio = audio / peak
        
        # Apply wav2vec2 feature extraction
        inputs = self.feature_extractor(
            audio,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=False
        )
        
        # Convert to bfloat16 to match model weights
        return {
            "input_values": inputs["input_values"].squeeze(0).to(torch.bfloat16),
            "label": torch.tensor(row["label"], dtype=torch.long)
        }

In [6]:
MODEL_NAME = "facebook/wav2vec2-base"
device     = torch.device("cuda")

print("Loading feature extractor...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)

print("Loading model...")
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)

# Freeze the CNN feature extractor
for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False

# Freeze the bottom 6 transformer layers (half of 12)
for i in range(6):
    for param in model.wav2vec2.encoder.layers[i].parameters():
        param.requires_grad = False

# Move model to GPU then convert weights to bfloat16
# bfloat16 is natively supported on Blackwell (sm_120)
# and avoids the type mismatch error with autocast
model = model.to(device)
model = model.to(torch.bfloat16)

# Allow TF32 — additional stability on Blackwell architecture
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(f"Model loaded on {device}")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")
print(f"Frozen parameters:    {total - trainable:,} / {total:,}")

Loading feature extractor...


Loading model...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda
Trainable parameters: 47,841,410 / 94,569,090
Frozen parameters:    46,727,680 / 94,569,090


In [7]:
from torch.utils.data import WeightedRandomSampler

# Create dataset objects
train_dataset = ASVspoofDataset(train_df, feature_extractor)
dev_dataset   = ASVspoofDataset(dev_df,   feature_extractor)

# Handle class imbalance using a weighted sampler
# Without this the model would see 9x more fake than real
# and would learn to just predict fake for everything
class_counts  = train_df["label"].value_counts().sort_index().values  # [bonafide_count, spoof_count]
class_weights = 1.0 / class_counts  # Rarer class gets higher weight
sample_weights = train_df["label"].map({0: class_weights[0], 1: class_weights[1]}).values
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# DataLoaders — these feed batches of audio to the model during training
train_loader = DataLoader(
    train_dataset,
    batch_size=8,          # 8 samples per batch — suits your GPU memory
    sampler=sampler,       # Use weighted sampler instead of shuffle
    num_workers=0          # Keep at 0 on Windows to avoid multiprocessing issues
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

print(f"Training batches:   {len(train_loader)}")
print(f"Development batches: {len(dev_loader)}")

Training batches:   3173
Development batches: 3106


c:\Users\tkell\anaconda3\envs\deepfake\Lib\site-packages\torch\utils\data\sampler.py:264: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  weights_tensor = torch.as_tensor(weights, dtype=torch.double)


In [8]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from sklearn.metrics import roc_auc_score

# AdamW optimiser
optimiser = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=0.01
)

# Linear warmup scheduler
total_steps  = len(train_loader) * 3
warmup_steps = int(0.1 * total_steps)
scheduler = LinearLR(
    optimiser,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_steps
)

def evaluate(model, loader, device):
    """Run model on dev set and return loss, accuracy and ROC-AUC."""
    model.eval()
    total_loss, correct, all_labels, all_probs = 0, 0, [], []

    with torch.no_grad():
        for batch in loader:
            # Convert input to bfloat16 to match model weights
            input_values = batch["input_values"].to(device)
            labels       = batch["label"].to(device)

            outputs = model(input_values=input_values, labels=labels)
            total_loss += outputs.loss.item()

            # Convert to float32 before numpy — bfloat16 is not supported by numpy
            probs  = torch.softmax(outputs.logits, dim=-1).to(torch.float32)
            preds  = probs.argmax(dim=-1)
            correct += (preds == labels).sum().item()

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset)
    roc_auc  = roc_auc_score(all_labels, all_probs)
    return avg_loss, accuracy, roc_auc

print("Optimiser and evaluation function ready")
print(f"Total training steps: {total_steps}")
print(f"Warmup steps:         {warmup_steps}")

Optimiser and evaluation function ready
Total training steps: 9519
Warmup steps:         951


In [9]:
from torch.amp import autocast
from sklearn.metrics import roc_auc_score

EPOCHS       = 3
EVAL_STEPS   = 200
SAVE_DIR     = r"C:\deepfake-project\models\wav2vec2_finetuned"
best_roc_auc = 0.0
patience     = 0
PATIENCE_MAX = 3

os.makedirs(SAVE_DIR, exist_ok=True)

print("Starting training...")
print(f"Evaluating every {EVAL_STEPS} steps — best model saved to {SAVE_DIR}")
print("-" * 60)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    step       = 0

    for batch in train_loader:
        input_values = batch["input_values"].to(device)
        labels       = batch["label"].to(device)

        # Forward pass wrapped in autocast — uses bfloat16 which is
        # natively supported on Blackwell (sm_120) unlike float16
        with autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(input_values=input_values, labels=labels)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        scheduler.step()
        optimiser.zero_grad()

        epoch_loss += loss.item()
        step       += 1

        if step % 50 == 0:
            avg_loss = epoch_loss / step
            print(f"Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {avg_loss:.4f}")

        if step % EVAL_STEPS == 0:
            dev_loss, dev_acc, dev_roc = evaluate(model, dev_loader, device)
            print(f"\n>>> Eval @ step {step} | Loss: {dev_loss:.4f} | Acc: {dev_acc:.4f} | ROC-AUC: {dev_roc:.4f}")

            if dev_roc > best_roc_auc:
                best_roc_auc = dev_roc
                patience     = 0
                model.save_pretrained(SAVE_DIR)
                feature_extractor.save_pretrained(SAVE_DIR)
                print(f"    New best model saved (ROC-AUC: {best_roc_auc:.4f})")
            else:
                patience += 1
                print(f"    No improvement — patience {patience}/{PATIENCE_MAX}")
                if patience >= PATIENCE_MAX:
                    print("\nEarly stopping triggered")
                    break

            model.train()

    if patience >= PATIENCE_MAX:
        break

    print(f"\nEpoch {epoch+1} complete | Avg loss: {epoch_loss/len(train_loader):.4f}\n")

print("-" * 60)
print(f"Training complete | Best ROC-AUC: {best_roc_auc:.4f}")
print(f"Best model saved to: {SAVE_DIR}")

Starting training...
Evaluating every 200 steps — best model saved to C:\deepfake-project\models\wav2vec2_finetuned
------------------------------------------------------------
Epoch 1 | Step 50/3173 | Loss: 0.6946
Epoch 1 | Step 100/3173 | Loss: 0.6930
Epoch 1 | Step 150/3173 | Loss: 0.6878
Epoch 1 | Step 200/3173 | Loss: 0.6677

>>> Eval @ step 200 | Loss: 0.7119 | Acc: 0.6018 | ROC-AUC: 0.9516


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    New best model saved (ROC-AUC: 0.9516)
Epoch 1 | Step 250/3173 | Loss: 0.6116
Epoch 1 | Step 300/3173 | Loss: 0.5552
Epoch 1 | Step 350/3173 | Loss: 0.5038
Epoch 1 | Step 400/3173 | Loss: 0.4567

>>> Eval @ step 400 | Loss: 0.6168 | Acc: 0.8109 | ROC-AUC: 0.9836


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    New best model saved (ROC-AUC: 0.9836)
Epoch 1 | Step 450/3173 | Loss: 0.4178
Epoch 1 | Step 500/3173 | Loss: 0.3834
Epoch 1 | Step 550/3173 | Loss: 0.3525
Epoch 1 | Step 600/3173 | Loss: 0.3287

>>> Eval @ step 600 | Loss: 0.3316 | Acc: 0.9125 | ROC-AUC: 0.9923


...

    New best model saved (ROC-AUC: 0.9993)
Epoch 1 | Step 2850/3173 | Loss: 0.0849
Epoch 1 | Step 2900/3173 | Loss: 0.0836
Epoch 1 | Step 2950/3173 | Loss: 0.0825
Epoch 1 | Step 3000/3173 | Loss: 0.0812

>>> Eval @ step 3000 | Loss: 0.1414 | Acc: 0.9679 | ROC-AUC: 0.9995


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    New best model saved (ROC-AUC: 0.9995)
Epoch 1 | Step 3050/3173 | Loss: 0.0799
Epoch 1 | Step 3100/3173 | Loss: 0.0786
Epoch 1 | Step 3150/3173 | Loss: 0.0774

Epoch 1 complete | Avg loss: 0.0769

Epoch 2 | Step 50/3173 | Loss: 0.0020
Epoch 2 | Step 100/3173 | Loss: 0.0031
Epoch 2 | Step 150/3173 | Loss: 0.0028
Epoch 2 | Step 200/3173 | Loss: 0.0039

>>> Eval @ step 200 | Loss: 0.1047 | Acc: 0.9762 | ROC-AUC: 0.9997


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    New best model saved (ROC-AUC: 0.9997)
Epoch 2 | Step 250/3173 | Loss: 0.0034
Epoch 2 | Step 300/3173 | Loss: 0.0031
Epoch 2 | Step 350/3173 | Loss: 0.0028
Epoch 2 | Step 400/3173 | Loss: 0.0026

>>> Eval @ step 400 | Loss: 0.0557 | Acc: 0.9877 | ROC-AUC: 0.9996
    No improvement — patience 1/3
Epoch 2 | Step 450/3173 | Loss: 0.0049
Epoch 2 | Step 500/3173 | Loss: 0.0046
Epoch 2 | Step 550/3173 | Loss: 0.0048
Epoch 2 | Step 600/3173 | Loss: 0.0046

>>> Eval @ step 600 | Loss: 0.1044 | Acc: 0.9775 | ROC-AUC: 0.9994
    No improvement — patience 2/3
Epoch 2 | Step 650/3173 | Loss: 0.0043
Epoch 2 | Step 700/3173 | Loss: 0.0042
Epoch 2 | Step 750/3173 | Loss: 0.0045
Epoch 2 | Step 800/3173 | Loss: 0.0043

>>> Eval @ step 800 | Loss: 0.0620 | Acc: 0.9870 | ROC-AUC: 0.9996
    No improvement — patience 3/3

Early stopping triggered
------------------------------------------------------------
Training complete | Best ROC-AUC: 0.9997
Best model saved to: C:\deepfake-project\models\wav2vec

In [10]:
# Load the saved model from disk
saved_model = Wav2Vec2ForSequenceClassification.from_pretrained(SAVE_DIR)
saved_extractor = Wav2Vec2FeatureExtractor.from_pretrained(SAVE_DIR)

saved_model = saved_model.to(device)
saved_model = saved_model.to(torch.bfloat16)
saved_model.eval()

def predict(audio_path):
    # Load and preprocess audio
    audio, sr = sf.read(audio_path)
    audio = audio.astype(np.float32)

    # Pad or truncate to 4 seconds
    if len(audio) >= MAX_SAMPLES:
        audio = audio[:MAX_SAMPLES]
    else:
        audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))

    # Normalise
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak

    # Feature extraction
    inputs = saved_extractor(
        audio,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding=False
    )

    input_values = inputs["input_values"].to(device).to(torch.bfloat16)

    with torch.no_grad():
        outputs = saved_model(input_values=input_values)
        probs = torch.softmax(outputs.logits, dim=-1).to(torch.float32)

    return {
        "real":    round(probs[0][0].item(), 4),
        "fake":    round(probs[0][1].item(), 4),
        "verdict": "REAL" if probs[0][0] > probs[0][1] else "FAKE"
    }

# Test on one real and one fake file from the dev set
real_file = dev_df[dev_df["label"] == 0].iloc[0]["path"]
fake_file = dev_df[dev_df["label"] == 1].iloc[0]["path"]

print("Testing saved model...")
print(f"\nReal audio: {os.path.basename(real_file)}")
print(predict(real_file))

print(f"\nFake audio: {os.path.basename(fake_file)}")
print(predict(fake_file))

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

Testing saved model...

Real audio: LA_D_1047731.flac
{'real': 1.0, 'fake': 0.0006, 'verdict': 'REAL'}

Fake audio: LA_D_1008730.flac
{'real': 0.001, 'fake': 1.0, 'verdict': 'FAKE'}
